# 10.3 上下文压缩 (Context Compression)

上下文压缩在保持关键信息的前提下减少输入token数量，是处理超长文本的重要策略。

## 压缩方法总览

| 方法 | 压缩比 | 信息损失 | 计算成本 | 适用场景 |
|------|--------|----------|----------|----------|
| 滑动窗口摘要 | 高 | 中 | 低 | 长文档 |
| KV Cache压缩 | 中 | 低 | 低 | 推理加速 |
| 提示压缩 | 中 | 中 | 低 | 减少API成本 |
| 分层压缩 | 高 | 中 | 中 | 超长文本 |
| 检索增强 | 变化 | 低 | 中 | 知识密集型 |

## 1. 滑动窗口摘要

滑动窗口摘要将长文本分段摘要，再拼接摘要作为压缩后的上下文。这是最简单也最实用的长文本压缩方法。

### 流程
1. 将长文本切分为固定大小的窗口
2. 对每个窗口生成摘要
3. 拼接所有摘要作为压缩上下文
4. 可选：对摘要再次摘要（层次化）

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class SlidingWindowCompressor:
    def __init__(self, window_size=64, stride=48, compression_ratio=0.3):
        self.window_size = window_size
        self.stride = stride
        self.compression_ratio = compression_ratio

    def compress(self, sequence, summary_model=None):
        B, T, D = sequence.shape
        summaries = []
        for start in range(0, T, self.stride):
            end = min(start + self.window_size, T)
            window = sequence[:, start:end, :]
            compressed_len = max(1, int((end - start) * self.compression_ratio))
            summary = F.adaptive_avg_pool1d(
                window.transpose(1, 2), compressed_len
            ).transpose(1, 2)
            summaries.append(summary)
        return torch.cat(summaries, dim=1)

class HierarchicalCompressor:
    def __init__(self, window_size=64, compression_ratio=0.3, n_levels=3):
        self.window_size = window_size
        self.compression_ratio = compression_ratio
        self.n_levels = n_levels

    def compress(self, sequence):
        current = sequence
        levels = [current]
        for level in range(self.n_levels):
            B, T, D = current.shape
            if T <= self.window_size:
                break
            n_windows = (T + self.window_size - 1) // self.window_size
            padded_len = n_windows * self.window_size
            if T < padded_len:
                padding = torch.zeros(B, padded_len - T, D, device=sequence.device)
                current = torch.cat([current, padding], dim=1)
            windows = current.reshape(B, n_windows, self.window_size, D)
            compressed_len = max(1, int(self.window_size * self.compression_ratio))
            compressed = F.adaptive_avg_pool1d(
                windows.reshape(B * n_windows, self.window_size, D).transpose(1, 2),
                compressed_len
            ).transpose(1, 2).reshape(B, n_windows, compressed_len, D)
            current = compressed.reshape(B, -1, D)
            levels.append(current)
        return levels

seq = torch.randn(1, 512, 64)

sw_compressor = SlidingWindowCompressor(window_size=64, stride=48, compression_ratio=0.3)
compressed = sw_compressor.compress(seq)

h_compressor = HierarchicalCompressor(window_size=64, compression_ratio=0.3, n_levels=3)
levels = h_compressor.compress(seq)

print('=== Context Compression ===')
print(f'Original sequence: {seq.shape}')
print(f'\nSliding window compressed: {compressed.shape}')
print(f'  Compression ratio: {compressed.shape[1]/seq.shape[1]:.1%}')

print(f'\nHierarchical compression levels:')
for i, level in enumerate(levels):
    print(f'  Level {i}: {level.shape[1]} tokens ({level.shape[1]/seq.shape[1]:.1%} of original)')

print(f'\nKey: Sliding window is simple and effective.')
print(f'Hierarchical compression creates multi-resolution representations.')

## 2. KV Cache压缩

KV Cache是LLM推理的显存瓶颈，压缩KV Cache可以显著降低显存使用和推理成本。

### KV Cache压缩方法
- **Token淘汰**：丢弃不重要的KV对（如H2O）
- **Token合并**：合并相似的KV对（如CaM）
- **量化**：将KV Cache从FP16量化到INT8/INT4
- **分组**：GQA/MQA减少KV头数

### H2O（Heavy-Hitter Oracle）
核心观察：少量关键token（heavy hitter）贡献了大部分注意力分数。
策略：保留最近token + 累计注意力最高的token。

In [ ]:
class KVCacheCompressor:
    def __init__(self, budget_ratio=0.5, recent_window=16):
        self.budget_ratio = budget_ratio
        self.recent_window = recent_window

    def h2o_evict(self, keys, values, attention_scores):
        B, T, H, D = keys.shape
        budget = max(self.recent_window, int(T * self.budget_ratio))

        if T <= budget:
            return keys, values

        cumulative_attn = attention_scores.sum(dim=(0, 1, 3))
        recent_start = T - self.recent_window
        cumulative_attn[recent_start:] = float('inf')

        _, top_indices = cumulative_attn.topk(budget - self.recent_window)
        recent_indices = torch.arange(recent_start, T, device=keys.device)
        keep_indices = torch.cat([top_indices, recent_indices]).sort()[0]

        return keys[:, keep_indices], values[:, keep_indices]

    def token_merging(self, keys, values, similarity_threshold=0.9):
        B, T, H, D = keys.shape
        merged_keys = [keys[:, 0:1]]
        merged_values = [values[:, 0:1]]
        counts = [torch.ones(B, 1, 1, 1, device=keys.device)]

        for i in range(1, T):
            sim = F.cosine_similarity(
                keys[:, i:i+1].flatten(2),
                merged_keys[-1].flatten(2), dim=-1
            ).mean()
            if sim > similarity_threshold:
                n = counts[-1]
                merged_keys[-1] = (merged_keys[-1] * n + keys[:, i:i+1]) / (n + 1)
                merged_values[-1] = (merged_values[-1] * n + values[:, i:i+1]) / (n + 1)
                counts[-1] = n + 1
            else:
                merged_keys.append(keys[:, i:i+1])
                merged_values.append(values[:, i:i+1])
                counts.append(torch.ones(B, 1, 1, 1, device=keys.device))

        return torch.cat(merged_keys, dim=1), torch.cat(merged_values, dim=1)

    def quantize_kv(self, keys, values, n_bits=8):
        def quantize(tensor, n_bits):
            max_val = tensor.abs().amax(dim=-1, keepdim=True)
            scale = max_val / (2 ** (n_bits - 1) - 1)
            quantized = torch.clamp(torch.round(tensor / scale), -(2 ** (n_bits - 1)), 2 ** (n_bits - 1) - 1)
            return quantized * scale
        return quantize(keys, n_bits), quantize(values, n_bits)

compressor = KVCacheCompressor(budget_ratio=0.5, recent_window=16)

keys = torch.randn(1, 128, 4, 16)
values = torch.randn(1, 128, 4, 16)
attn_scores = torch.softmax(torch.randn(1, 4, 10, 128), dim=-1)

print('=== KV Cache Compression ===')
print(f'Original KV: keys={keys.shape}, values={values.shape}')
print(f'Original memory: {keys.numel() * 2 + values.numel() * 2} bytes (FP16)')

comp_keys, comp_values = compressor.h2o_evict(keys, values, attn_scores)
print(f'\nH2O compressed: keys={comp_keys.shape}, values={comp_values.shape}')
print(f'Memory savings: {1 - comp_keys.shape[1]/keys.shape[1]:.1%}')

merged_keys, merged_values = compressor.token_merging(keys, values, similarity_threshold=0.8)
print(f'\nToken merging: keys={merged_keys.shape}, values={merged_values.shape}')
print(f'Memory savings: {1 - merged_keys.shape[1]/keys.shape[1]:.1%}')

q_keys, q_values = compressor.quantize_kv(keys, values, n_bits=8)
quant_error_k = (keys - q_keys).abs().mean().item()
print(f'\nINT8 quantization error: keys={quant_error_k:.6f}')
print(f'Memory savings: 50% (16bit -> 8bit)')

print(f'\nKey: H2O keeps important tokens + recent tokens.')
print(f'Token merging combines similar tokens. Quantization reduces precision.')

## 3. 提示压缩（Prompt Compression）

提示压缩通过删除或压缩提示中的冗余信息，减少输入token数量，降低API成本和延迟。

### 方法
- **LLMLingua**：用小模型评估token重要性，删除不重要的token
- **Selective Context**：基于信息熵选择保留哪些内容
- **LongLLMLingua**：结合问题感知的提示压缩

### 压缩流程
1. 用小模型计算每个token的困惑度
2. 低困惑度token = 冗余信息，可以删除
3. 高困惑度token = 关键信息，必须保留
4. 按目标压缩比删除token

In [ ]:
class PromptCompressor:
    def __init__(self, compression_ratio=0.5):
        self.compression_ratio = compression_ratio

    def compute_importance(self, tokens, model=None):
        importance = torch.rand(tokens.shape[0], tokens.shape[1])
        importance[:, :3] = 1.0
        importance[:, -3:] = 1.0
        return importance

    def compress(self, tokens, importance_scores=None):
        if importance_scores is None:
            importance_scores = self.compute_importance(tokens)

        B, T = tokens.shape
        keep_count = max(1, int(T * (1 - self.compression_ratio)))
        _, keep_indices = importance_scores.topk(keep_count, dim=-1)
        keep_indices = keep_indices.sort(dim=-1).values

        compressed = torch.gather(tokens, 1, keep_indices)
        return compressed, keep_indices

    def compress_with_structure(self, tokens, structure_mask=None):
        B, T = tokens.shape
        importance = self.compute_importance(tokens)

        if structure_mask is not None:
            importance = importance * structure_mask

        keep_count = max(1, int(T * (1 - self.compression_ratio)))
        _, keep_indices = importance.topk(keep_count, dim=-1)
        keep_indices = keep_indices.sort(dim=-1).values
        compressed = torch.gather(tokens, 1, keep_indices)
        return compressed

compressor = PromptCompressor(compression_ratio=0.5)
tokens = torch.randint(0, 1000, (2, 50))

compressed, keep_indices = compressor.compress(tokens)

print('=== Prompt Compression ===')
print(f'Original tokens: {tokens.shape[1]}')
print(f'Compressed tokens: {compressed.shape[1]}')
print(f'Compression ratio: {1 - compressed.shape[1]/tokens.shape[1]:.0%}')
print(f'\nKept positions (sample 0): {keep_indices[0].tolist()}')

for ratio in [0.2, 0.4, 0.6, 0.8]:
    comp = PromptCompressor(compression_ratio=ratio)
    c, _ = comp.compress(tokens)
    print(f'  Ratio {ratio:.0%}: {tokens.shape[1]} -> {c.shape[1]} tokens')

print(f'\nKey: Prompt compression removes low-importance tokens.')
print(f'First/last tokens and structural tokens should always be preserved.')
print(f'Compression ratio 30-50% typically preserves most task performance.')

## 4. 上下文压缩工业实践

### 方法选择
| 场景 | 推荐方法 | 原因 |
|------|----------|------|
| 长文档QA | 滑动窗口摘要+检索 | 保留关键信息 |
| 推理加速 | KV Cache量化+淘汰 | 减少显存和延迟 |
| API成本优化 | 提示压缩 | 减少token数 |
| 超长文本 | 层次化压缩 | 多分辨率处理 |

### 最佳实践
1. **压缩比选择**：30-50%压缩比通常安全，>70%会显著损失信息
2. **关键信息保护**：系统提示、问题、格式要求不应被压缩
3. **评估驱动**：压缩前后对比任务性能，确保可接受
4. **动态压缩**：根据输入长度动态调整压缩比
5. **组合策略**：KV Cache量化 + Token淘汰 + 提示压缩组合使用